# Solaris-3 OCR · entrenamiento en Kaggle (GPU)

Genera el dataset sintético limpio, entrena `SolarisCharNet` (detector) y `GlyphNet` (clasificador 32×32),
exporta ambos a ONNX en CPU y verifica con `onnxruntime` (`CPUExecutionProvider`).

**Acelerador:** GPU T4/P100 · **Internet:** no necesario (el código llega como dataset privado).

Al terminar, descarga `/kaggle/working/artifacts/` y copia `model.onnx` y `classifier.onnx` a `public/models/`.

In [ ]:
import glob, os, shutil, subprocess, sys

# Locate the project either as a Kaggle dataset or as a cloned repo.
cands = glob.glob('/kaggle/input/*/repo') + glob.glob('/kaggle/input/*/wuwa-translator-ocr') + ['/kaggle/working/repo']
repo = next((c for c in cands if os.path.exists(os.path.join(c, 'scripts', 'kaggle_train.py'))), None)
assert repo, 'Project not found: attach the private dataset or clone the repo first'
work = '/kaggle/working/repo'
if os.path.abspath(repo) != work:
    shutil.copytree(repo, work, dirs_exist_ok=True)
os.chdir(work)
print('repo:', work)
print(subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")'], capture_output=True, text=True).stdout)

In [ ]:
# Full pipeline: dataset -> detector -> classifier -> ONNX -> CPU check.
!python scripts/kaggle_train.py --gpu --workers 4

!mkdir -p /kaggle/working/artifacts
!cp public/models/model.onnx public/models/classifier.onnx src/lib/model-meta.json /kaggle/working/artifacts/
!ls -lh /kaggle/working/artifacts

In [ ]:
# Quick sanity check on the generated validation split (no Holdout_5 here).
!python scripts/eval_holdout.py --dir training/micro/images/val --conf 0.3 || true